# ENTRO-CORE: Noise Sensitivity Analysis

## How does noise affect the hybrid controller?

In [ ]:
import sys
import os
import random
sys.path.insert(0, os.path.dirname(os.getcwd()))

from entro_core.hybrid_controller import HybridController
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
class NoisySystem:
    def __init__(self, psi_0=1.8, dpsi_0=0.3, noise_std=0.0):
        self.psi = psi_0
        self.dpsi = dpsi_0
        self.noise_std = noise_std
        self.history = []
    
    def step(self, u, dt=0.1):
        noise = random.gauss(0, self.noise_std)
        d2psi = -0.3 * self.dpsi - 0.5 * self.psi + u + noise
        self.dpsi += d2psi * dt
        self.psi += self.dpsi * dt
        self.history.append(self.psi)
        return self.psi
    
    def reset(self):
        self.psi = 1.8
        self.dpsi = 0.3
        self.history = []

In [ ]:
noise_levels = [0.0, 0.02, 0.05, 0.1, 0.15, 0.2]
results = {}

for noise in noise_levels:
    system = NoisySystem(noise_std=noise)
    controller = HybridController(threshold=1.7)
    
    for _ in range(200):
        result = controller.step(system.psi)
        system.step(result.u)
    
    results[noise] = system.history

In [ ]:
plt.figure(figsize=(12, 8))

for noise, traj in results.items():
    plt.plot(traj, label=f'σ = {noise}', linewidth=1.5)

plt.axhline(y=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
plt.xlabel('Time Step (dt=0.1s)')
plt.ylabel('Ψ (Entropy State)')
plt.title('Hybrid Controller: Noise Sensitivity')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
print("=" * 50)
print("NOISE SENSITIVITY RESULTS")
print("=" * 50)
for noise, traj in results.items():
    print(f"Noise σ = {noise:.2f} → Final Ψ = {traj[-1]:.3f}")
print("=" * 50)